<a href="https://colab.research.google.com/github/Ariyan-Sk/AI-fruit-classifier/blob/main/SVM_and_RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub
import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [3]:
muhammad0subhan_fruit_and_vegetable_disease_healthy_vs_rotten_path = kagglehub.dataset_download('muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten')

print('Data source import complete.')

print(os.listdir(muhammad0subhan_fruit_and_vegetable_disease_healthy_vs_rotten_path))

Using Colab cache for faster access to the 'fruit-and-vegetable-disease-healthy-vs-rotten' dataset.
Data source import complete.
['Fruit And Vegetable Diseases Dataset']


In [4]:
# DATA PATH
data_dir = os.path.join(
    muhammad0subhan_fruit_and_vegetable_disease_healthy_vs_rotten_path,
    "Fruit And Vegetable Diseases Dataset"
)

# IMAGE SETTINGS
img_height = 128
img_width = 128
batch_size = 32

# LOAD TRAIN DATASET
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# LOAD VALIDATION DATASET
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 29277 files belonging to 28 classes.
Using 23422 files for training.
Found 29277 files belonging to 28 classes.
Using 5855 files for validation.
Classes: ['Apple__Healthy', 'Apple__Rotten', 'Banana__Healthy', 'Banana__Rotten', 'Bellpepper__Healthy', 'Bellpepper__Rotten', 'Carrot__Healthy', 'Carrot__Rotten', 'Cucumber__Healthy', 'Cucumber__Rotten', 'Grape__Healthy', 'Grape__Rotten', 'Guava__Healthy', 'Guava__Rotten', 'Jujube__Healthy', 'Jujube__Rotten', 'Mango__Healthy', 'Mango__Rotten', 'Orange__Healthy', 'Orange__Rotten', 'Pomegranate__Healthy', 'Pomegranate__Rotten', 'Potato__Healthy', 'Potato__Rotten', 'Strawberry__Healthy', 'Strawberry__Rotten', 'Tomato__Healthy', 'Tomato__Rotten']


In [5]:
# NORMALIZATION
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
# CONVERT DATASET INTO NUMPY ARRAYS

X_train = []
y_train = []

for images, labels in train_ds:

    # Flatten images
    images = tf.reshape(images, (images.shape[0], -1))

    X_train.append(images.numpy())
    y_train.append(labels.numpy())

X_train = np.concatenate(X_train, axis=0)
y_train = np.concatenate(y_train, axis=0)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

# VALIDATION DATA

X_val = []
y_val = []

for images, labels in val_ds:

    images = tf.reshape(images, (images.shape[0], -1))

    X_val.append(images.numpy())
    y_val.append(labels.numpy())

X_val = np.concatenate(X_val, axis=0)
y_val = np.concatenate(y_val, axis=0)

print("X_val shape:", X_val.shape)

In [ ]:
# RANDOM FOREST MODEL

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

print("Training Random Forest...")

rf_model.fit(X_train, y_train)

# PREDICTION
rf_pred = rf_model.predict(X_val)

# ACCURACY
rf_accuracy = accuracy_score(y_val, rf_pred)

print("Random Forest Accuracy:", rf_accuracy)

In [ ]:
# SVM MODEL

svm_model = SVC(
    kernel='rbf',
    probability=True
)

print("Training SVM...")

svm_model.fit(X_train, y_train)

# PREDICTION
svm_pred = svm_model.predict(X_val)

# ACCURACY
svm_accuracy = accuracy_score(y_val, svm_pred)

print("SVM Accuracy:", svm_accuracy)

In [ ]:
# CLASSIFICATION REPORT

print("\nRandom Forest Report\n")
print(classification_report(y_val, rf_pred))

print("\nSVM Report\n")
print(classification_report(y_val, svm_pred))

In [ ]:
# TESTING WITH SINGLE IMAGE

sample_path = os.path.join(
    data_dir,
    "Carrot__Healthy",
    os.listdir(os.path.join(data_dir, "Carrot__Healthy"))[0]
)

print(sample_path)

img = tf.keras.utils.load_img(
    sample_path,
    target_size=(128,128)
)

img_array = tf.keras.utils.img_to_array(img)

# Normalize
img_array = img_array / 255.0

# Flatten
img_array = img_array.reshape(1, -1)

# RANDOM FOREST PREDICTION
rf_prediction = rf_model.predict(img_array)

print(
    "Random Forest Predicted Class:",
    class_names[rf_prediction[0]]
)

# SVM PREDICTION
svm_prediction = svm_model.predict(img_array)

print(
    "SVM Predicted Class:",
    class_names[svm_prediction[0]]
)